# What Is an AI Agent?

This is the code version of the agents course. If you took the [n8n course](https://ezponda.github.io/ai-agents-course/), you built these ideas on a canvas; here you build them in Python — and you get to see, and control, everything the canvas did for you.

Because you're going to write the code, this notebook goes deeper than "an agent uses tools in a loop." We'll pin down three ideas that decide how every later notebook is written:

1. A model call is a **stateless function**: text in, text out, nothing remembered.
2. **Tool calling is a protocol**, not a capability the model has on its own.
3. Two hard constraints — **non-determinism** and a finite **context window** — shape every design choice.

There's no code to run here; it's the map for the rest of Block 0.

## A single LLM call, precisely

Strip away the frameworks and a language-model call is one thing: a function that takes some text and returns some text.

```
┌─────────────────┐          ┌─────────────────┐          ┌─────────────────┐
│   Your prompt   │────────▶│       LLM       │────────▶│  Text response  │
└─────────────────┘          └─────────────────┘          └─────────────────┘
```

Two properties of that function matter more than anything else, and beginners miss both:

- **It is stateless.** The model keeps nothing between calls. Call it twice and the second call has no idea the first happened. Every "memory" you'll ever build is you re-sending the earlier text yourself.
- **It only produces text.** It cannot fetch a web page, run a query, or add two numbers reliably. It predicts the next tokens of text — that's the whole mechanism. Everything an agent *does* in the world is your code acting on the model's text.

**Great for:** translation, summaries, rewriting, classification — self-contained text tasks.

**Limitations that follow directly from those two properties:**
- No actions and no fresh facts (it can't reach outside the prompt).
- One shot — if it's wrong or invents a detail, nothing notices.
- Everything crammed into one prompt tends to lower quality, not raise it.

## Workflows: you control the steps

The first fix is to stop writing one giant prompt and instead chain several small, focused steps that *you* wire together:

```
                             YOU design this sequence
                                      │
        ┌─────────────────────────────┼─────────────────────────────┐
        ▼                             ▼                             ▼
┌───────────────┐           ┌───────────────┐           ┌───────────────┐
│    STEP 1     │──────────▶│    STEP 2     │──────────▶│    STEP 3     │
│  Extract info │           │  Improve it   │           │  Write draft  │
└───────────────┘           └───────────────┘           └───────────────┘
```

**Key characteristics:**
- You decide the steps in advance; the flow runs the same way every time.
- Each step is simpler, so each step is more reliable and easier to debug.
- Tools are optional, and *you* choose them at design time — not the model.

**In code:** a workflow is just ordinary Python — a few calls in a row, an `if` to branch, an `asyncio.gather` to fan out. You'll write these in notebook 0.4 and see they need no framework at all. This is the level most "AI features" in production actually live at.

## Agents: the model controls the steps

An agent hands that control to the model. You give it a goal and some tools; it decides what to do next, in a loop, until it's done:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              THE AGENT LOOP                                  │
│                                                                             │
│    ┌─────────┐      ┌─────────┐      ┌─────────┐                            │
│    │  THINK  │─────▶│   ACT   │─────▶│ OBSERVE │                            │
│    │         │      │         │      │         │                            │
│    │ "What   │      │  Call   │      │ "Do I   │                            │
│    │  do I   │      │  a tool │      │  have   │──── no, loop ───┐          │
│    │  need?" │      │         │      │ enough?"│                 │          │
│    └─────────┘      └────┬────┘      └────┬────┘                 │          │
│         ▲                │                │ yes                  │          │
│         │                ▼                ▼                      │          │
│         │          ┌──────────┐    ┌─────────────┐              │          │
│         │          │  TOOLS   │    │FINAL ANSWER │              │          │
│         │          │ search   │    └─────────────┘              │          │
│         │          │ calc     │                                 │          │
│         │          │ database │                                 │          │
│         │          └──────────┘                                 │          │
│         └─────────────────────────────────────────────────────┘          │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Key characteristics:**
- The model decides the next step, at runtime, based on what it has seen so far.
- It can call **tools** (search, a calculator, an API, a database, code).
- It **loops** until the task is done, and can **adapt** when a step fails.

That flexibility is the whole point — and also the whole problem, because it trades away predictability. Reclaiming that predictability (guardrails, evals, state) is what Blocks 1 and 2 are about.

## The key question: who decides?

| Approach | Who picks the steps? | Who picks the tools? | Loops? | Best for |
|----------|----------------------|----------------------|--------|----------|
| **Single LLM call** | You (one prompt) | No tools | No | Quick text tasks |
| **Workflow** | You (in advance) | You (at design time) | No | Predictable, structured tasks |
| **Agent** | The model (dynamic) | The model (at runtime) | Yes | Open-ended, ambiguous tasks |

**Rule of thumb:** if you can write down all the steps ahead of time, use a workflow. If the model has to figure the steps out on the fly, use an agent.

```{note}
Reach for the simplest row that solves the problem. Agents are the most capable and the least predictable option, and they cost the most tokens. We build workflows first (0.4) precisely so you can tell when an agent is overkill.
```

## The loop up close: ReAct

The loop in the diagram has a name — **ReAct**, for *Reason + Act* (from the [2022 paper](https://arxiv.org/abs/2210.03629)). One turn is: reason about what's needed, act by calling a tool, observe the result, then loop or answer. In pseudocode — and notebook 0.3 is barely longer than this:

```python
messages = [system_prompt, user_message]

while True:
    response = model.call(messages, tools=my_tools)   # THINK / reason
    if response.tool_calls:                            # the model asked for a tool
        result = run_tool(response.tool_calls[0])      # ACT — your code runs it
        messages += [response, result]                 # OBSERVE — feed it back, loop
    else:
        return response.content                        # no tool wanted → final answer
```

Two subtleties that only become obvious once you write it:

- **"Wants a tool" is a real signal from the API**, not a guess. The model's reply comes back either as ordinary content or as a structured `tool_calls` field. Your loop branches on which.
- **The termination condition is "no tool requested."** The loop ends when the model replies with plain text instead of another tool call. A missing or wrong stop condition is the classic way an agent runs forever (or one step).

Read the source of a real coding agent and the shape is the same: the loop is maybe 5% of the code; the other 95% is managing context, running tools safely, and handling errors. Block 0 builds the 5% so the 95% makes sense later.

## Tool calling is a protocol, not magic

This is the single idea that makes agents click, and it's worth slowing down for. **The model cannot run anything itself.** It can't open a URL, hit your database, or execute Python. All it can do is emit text — including text that *requests* a tool. Running the tool is your job. Tool calling is a hand-off in four moves:

```
  1. You describe the tools in the request   (name, what it does, its arguments)
  2. Instead of answering, the model emits a structured call:
         { "name": "get_weather", "arguments": { "city": "Bilbao" } }
  3. YOUR code runs the real function and gets a result       ("18C, rain")
  4. You send that result back as a new message; the model continues

     ┌──────────┐   "call get_weather(Bilbao)"   ┌────────────┐
     │  Model   │ ─────────────────────────────▶ │  Your code │
     │ (only    │                                │  runs the  │
     │  text)   │ ◀───────────────────────────── │  function  │
     └──────────┘      "result: 18C, rain"        └────────────┘
```

The model never touches the outside world. It only produces text, including the text that asks for an action; your code is what acts and what enforces limits. That separation is also your main safety boundary — the model can *ask* to delete a file, but it can only do it if you wrote code that lets it.

## What an agent is made of

Every capability later in the course slots into one of these parts:

```
┌─────────────────────────────────────────────────────────────────────┐
│                              AI AGENT                                │
│   ┌───────────┐   ┌───────────┐   ┌───────────┐   ┌───────────┐     │
│   │    LLM    │   │   TOOLS   │   │  MEMORY   │   │ REASONING │     │
│   │  (brain)  │   │ (actions) │   │ (context) │   │  (logic)  │     │
│   └───────────┘   └───────────┘   └───────────┘   └───────────┘     │
│   understands     acts on the     remembers the    plans the        │
│   language        outside world   conversation     next step        │
└─────────────────────────────────────────────────────────────────────┘
```

| Component | What it does | Where in this course |
|-----------|--------------|----------------------|
| **LLM** | Reasons and generates text | 0.1 |
| **Tools** | Functions the model can request: search, code, APIs | 0.3, 1.2 |
| **Memory** | The message history / state you carry across turns | 0.6, 1.4, 2.3 |
| **Reasoning** | Planning and deciding across the loop | 0.3, 0.5, 2.5 |

Underneath all four is **context** — what you actually place in front of the model on each turn. It isn't a component; it's the thing that determines whether the others work. "Context engineering" — deciding what to include, what to leave out, and how to phrase it — is a thread we return to in every notebook.

## Agentic design patterns

Most agent systems are combinations of four well-known patterns. You'll build each one, in code:

| Pattern | What it means | Where |
|---------|---------------|-------|
| **Tool use** | The agent calls functions to act and to fetch facts | 0.3, 1.2 |
| **Reflection** | The agent reviews and improves its own output | 0.5, 2.5 |
| **Planning** | The agent breaks a task into steps before acting | 0.4, 2.5 |
| **Multi-agent** | Several specialized agents collaborate | 2.6 |

**Tool use is the foundation** — once you understand how an agent calls tools, the other three are variations on the same loop. That's why Block 0 spends its longest notebook (0.3) there.

```{note}
These four patterns come from Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents), the same reference the n8n course used. It's the best short read in the field — worth 20 minutes.
```

## Two hard truths for engineers

Two facts about LLMs constrain every agent you'll build. They aren't bugs; they're the material you're working with.

**1. Non-determinism.** The same input can produce different output. Models sample from a probability distribution, so runs vary — which breaks the mental model you have from normal code. You can turn the randomness down (`temperature=0`, in 0.1) but not fully off, and you can never assume "it worked once" means "it works." This is *the* reason Block 1 spends a whole notebook on **evals** (1.6): with non-deterministic software, you measure quality statistically, you don't eyeball it.

**2. The context window is a hard budget.** Everything the model sees on a turn — system prompt, history, tool results, retrieved documents — must fit in a fixed token limit. It's finite and you pay per token. This single constraint is why three later topics exist at all: **memory** management (you can't keep everything — 0.6), **RAG** (fetch only the relevant documents instead of pasting them all — 1.7), and **context engineering** (spend the budget well).

Here's the same idea as a table, since it drives so much of the course:

| What LLMs struggle with | Why | How this course handles it |
|-------------------------|-----|-----------------------------|
| Exact math, counting | They predict text, not compute | Give them a calculator/code tool (0.3, projects) |
| Fresh or private facts | Training data has a cutoff | Tools + RAG (1.2, 1.7) |
| Remembering long chats | Finite context window | Manage memory; summarize (0.6, 2.3) |
| Following rules exactly | Non-deterministic output | Guardrails + validation + evals (1.5, 1.6) |
| Long multi-step plans | Hard to track many steps | Decompose into workflows / a graph (0.4, 2.x) |

## The bigger map, and what you'll build

Research programs like UC Berkeley's [Agentic AI course](https://agenticai-learning.org/) organize the field around a few pillars. This course covers each — here's where, alongside what you actually build:

| Pillar | You build | Notebook(s) |
|--------|-----------|-------------|
| Reasoning and planning | The ReAct loop; a reflection loop | 0.3, 0.5, 2.5 |
| Tool use | An agent that calls your Python functions | 0.3, 1.2 |
| Memory and retrieval | Chat memory, then a knowledge agent (RAG) | 0.6, 1.4, 1.7 |
| Multi-agent | A supervisor delegating to specialists | 2.6 |
| Evaluation | A test suite that scores your agent | 1.6 |
| Safety and security | Guardrails against bad input and prompt injection | 1.5 |
| Deployment | Your agent behind a real HTTP API | 3.1 |

**What's next:** in **0.1** you set up a model-agnostic environment — one variable to switch between Claude, GPT, Gemini, or Llama — make your first call, and *see* the statelessness and token budget from this notebook in real output.